In [1]:
# 1. Check the # & order of embeddings

import scanpy as sc
import numpy as np
import os
from sklearn.preprocessing import StandardScaler


In [3]:
team_files = {
    'scGPT': 'scGPT_results.h5ad',
    'Geneformer': 'Geneformer_results.h5ad',
    'scVI': 'scVI_results.h5ad'
}

keys = {
    'scGPT': 'X_scGPT',
    'Geneformer': 'X_Geneformer',
    'scVI': 'X_scVI'
}

In [4]:
def merge_h5ad_embeddings(file_dict, key_dict):
    all_embeds = []
    reference_barcodes = None
    
    print("🚀 Start merging...")

    for model_name, path in file_dict.items():
        if not os.path.exists(path):
            print(f"🚨 No file : {model_name} ({path})")
            return None
        
        # read h5ad
        adata = sc.read_h5ad(path)
        print(f"📦 {model_name} load done : {adata.shape}")
        
        # Check the barcode order 
        current_barcodes = adata.obs_names.values
        if reference_barcodes is None:
            reference_barcodes = current_barcodes
            print(f"✅ reference barcode setting done ({model_name})")
        else:
            if not np.array_equal(reference_barcodes, current_barcodes):
                print(f"❌ Error : cell order of {model_name} is different with reference.")
                return None
        
        # Extract embedding from obsm
        embed_key = key_dict[model_name]
        if embed_key in adata.obsm:
            all_embeds.append(adata.obsm[embed_key])
            print(f"   - '{embed_key}' Extraction complete: {adata.obsm[embed_key].shape}")
        else:
            print(f"🚨 Error : No '{embed_key}' key in {model_name} file.")
            return None

    # Feature Fusion
    # $$ \text{Combined} = [E_{scGPT} \mid E_{Geneformer} \mid E_{scVI}] $$
    fused_matrix = np.concatenate(all_embeds, axis=1)
    print(f"\n🔗 Merging complete ! Final dimension : {fused_matrix.shape}")
    
    return fused_matrix, reference_barcodes

In [5]:
result = merge_h5ad_embeddings(team_files, keys)

🚀 Start merging...
🚨 No file : scGPT (scGPT_results.h5ad)


In [ ]:
if result:
    fused_matrix, barcodes = result
    
    # Scaling
    scaler = StandardScaler()
    scaled_embedding = scaler.fit_transform(fused_matrix)
    print("⚖️ Scaling done.")

    # Save
    np.save('scaled_embedding.npy', scaled_embedding)
    final_adata = sc.AnnData(X=scaled_embedding, obs=dict(obs_names=barcodes))
    final_adata.write_h5ad('RNA_combined_final.h5ad')
    print("💾 Final file saved : scaled_embedding.npy, RNA_combined_final.h5ad")

In [ ]:
# 2. Projection Layer

import torch
import torch.nn as nn

In [ ]:
class RNAFusionEncoder(nn.Module):
    def __init__(self, input_dim, output_dim=512):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),      
            
            nn.Linear(1024, 512),
            nn.ReLU(),
            
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        return self.network(x)


In [ ]:
input_size = scaled_embedding.shape[1]
rna_fusion_model = RNAFusionEncoder(input_dim=input_size, output_dim=512)

In [ ]:
import h5py

h5_file_path = 'RNA_embeddings_final.h5'

with h5py.File(h5_file_path, 'w') as f:
    # embeddings
    f.create_dataset('embeddings', data=scaled_embedding)
    
    # cell_ids
    ascii_barcodes = [b.encode("utf-8") for b in barcodes]
    f.create_dataset('cell_ids', data=ascii_barcodes)

print(f"✅ Final file saved : {h5_file_path}")